# Fase 3b – Indice italiano per la ricerca nel browser

Aggiunge la versione italiana **senza modificare i file inglesi** gia' pubblicati: produce in
`/kaggle/working/web_assets_it` i file con suffisso `_it` (`chunks_it.json`, `embeddings_it.f16.bin`,
`config_it.json`, `eval_set_it.json`, `parity_it.json`), con per ogni chunk italiano la pagina e,
quando l'abbinamento e' esatto, l'id del chunk inglese corrispondente.

**Impostazioni:** GPU T4, Internet ON. **Input:** Dataset del parsing italiano e Dataset del parsing inglese.

In [ ]:
!pip install -q "qrt-rag[eval] @ git+https://github.com/mat126/rag-qrt-SolvencyII.git"

In [ ]:
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer

from qrt_rag.alignment import align_pages, alignment_report
from qrt_rag.chunking import build_chunks
from qrt_rag.evaluation import load_eval_set
from qrt_rag.index import PrefixedEncoder, embed_chunks
from qrt_rag.parsing import load_pages
from qrt_rag.web_export import export_web

WORK = Path("/kaggle/working")
PARSED_IT = Path("/kaggle/input/datasets/mat126/parsing-rag-qrt-ivass")
PARSED_EN = Path("/kaggle/input/datasets/mat126/parsing-rag-qrt-eng")
OUT = WORK / "web_assets_it"

PDFS = {"en": "regulation_en.pdf", "it": "regulation_it.pdf"}   # stessi nomi gia' nello Space
MODEL = {   # stesso modello dell'inglese: nel browser se ne scarica uno solo
    "hf": "intfloat/multilingual-e5-small",
    "web": "Xenova/multilingual-e5-small",
    "q_prefix": "query: ",
    "p_prefix": "passage: ",
    "pooling": "mean",
}
EXPECTED_EN_CHUNKS = 7985   # i chunk inglesi gia' pubblicati: gli id devono coincidere
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

In [ ]:
chunks_it = build_chunks(load_pages(PARSED_IT), lang="it")
chunks_en = build_chunks(load_pages(PARSED_EN), lang="en")
cases_it = load_eval_set(lang="it")
print(f"Chunk italiani: {len(chunks_it)} | chunk inglesi: {len(chunks_en)} | casi: {len(cases_it)}")
assert len(chunks_en) == EXPECTED_EN_CHUNKS, (
    "I chunk inglesi non coincidono con quelli pubblicati: gli id per il testo affiancato "
    "sarebbero sbagliati. Riesporta anche l'inglese con fase3_export_web.ipynb.")

In [ ]:
# Per ogni chunk italiano: pagina nel PDF inglese e, se l'abbinamento e' esatto,
# l'id del chunk inglese (serve a mostrare affiancati i due testi)
ref_pages = align_pages(chunks_it, chunks_en)
_ = alignment_report(chunks_it, ref_pages)

In [ ]:
model = SentenceTransformer(MODEL["hf"], device=DEVICE)
E_it = embed_chunks(model, chunks_it, batch_size=64, prefix=MODEL["p_prefix"])

In [ ]:
# Le metriche stampate sono quelle Python sul set di valutazione italiano:
# il riferimento inglese e' hit@5 0,968, MRR 0,620
config_it = export_web(OUT, chunks_it, E_it, PrefixedEncoder(model, MODEL["q_prefix"]), MODEL, cases_it,
                       lang="it", pdfs=PDFS, ref_lang="en", ref_pages=ref_pages, suffix="_it")

In [ ]:
# Archivio da scaricare dal pannello Output: i file vanno in web/assets/ accanto a quelli inglesi
import shutil
shutil.make_archive(str(WORK / "web_assets_it"), "zip", OUT)
print("Creato", WORK / "web_assets_it.zip")